# Simple benchmark and embedding-based model tests

Copyright (c) jphall@gwu.edu 2026-27. Released under the MIT License; see the repository `LICENSE` file.

This notebook runs a small multiple-choice benchmark, reports uncertainty, and uses embeddings as a lightweight classifier for model-testing intents. The examples are inspectable teaching cases, not evidence for a deployment decision.

## 1. Configure the OpenAI client

The client loads `OPENAI_API_KEY` from the project-root `.env`; if it is unavailable, the notebook asks for it without echoing the value.

In [1]:
# 1. Configure the OpenAI client
import os
import re
from getpass import getpass
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from openai import OpenAI

# Walk upward so the notebook works from Jupyter or nbconvert.
project_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / '.env').exists()), Path.cwd())
load_dotenv(project_root / '.env')
api_key = os.getenv('OPENAI_API_KEY') or getpass('OpenAI API key: ')
client = OpenAI(api_key=api_key)
TEXT_MODEL = os.getenv('OPENAI_TEXT_MODEL', 'gpt-5-nano')
EMBEDDING_MODEL = os.getenv('OPENAI_EMBEDDING_MODEL', 'text-embedding-3-small')
print(f'Configured text model: {TEXT_MODEL}; embedding model: {EMBEDDING_MODEL}')

Configured text model: gpt-5-nano; embedding model: text-embedding-3-small


## 2. Define a transparent mini-benchmark

Each item contains the prompt, choices, answer key, and a slice label. Keeping these components together makes errors auditable.

In [2]:
# 2. Define a transparent mini-benchmark
questions = [
    {'slice': 'measurement', 'question': 'Which property asks whether repeated measurements are sufficiently consistent?', 'choices': ['A. Reliability', 'B. Novelty', 'C. Popularity', 'D. Compression'], 'answer': 'A'},
    {'slice': 'measurement', 'question': 'A test can be repeatable but consistently measure the wrong proxy. What is most directly missing?', 'choices': ['A. Latency', 'B. Validity', 'C. Throughput', 'D. Compression'], 'answer': 'B'},
    {'slice': 'testing-level', 'question': 'Which level deliberately asks motivated testers to defeat safeguards?', 'choices': ['A. Model testing', 'B. Red teaming', 'C. Field testing', 'D. Unit conversion'], 'answer': 'B'},
    {'slice': 'testing-level', 'question': 'Which level studies representative users and downstream actions in realistic settings?', 'choices': ['A. Field testing', 'B. Static linting', 'C. Tokenization', 'D. Pretraining'], 'answer': 'A'},
    {'slice': 'benchmark-risk', 'question': 'When public test items enter model-development data, what threatens the score?', 'choices': ['A. Contamination', 'B. Encryption', 'C. Quantization', 'D. Caching'], 'answer': 'A'},
    {'slice': 'benchmark-risk', 'question': 'Optimizing a metric until it stops representing the goal illustrates what risk?', 'choices': ['A. Goodhart pressure', 'B. Bagging', 'C. Token budgeting', 'D. Distillation'], 'answer': 'A'},
]
print(f'{len(questions)} benchmark items across {len({q["slice"] for q in questions})} slices')

6 benchmark items across 3 slices


## 3. Run the benchmark with a fixed protocol

The prompt requests one letter, and the parser accepts only `A`–`D`. Raw text is retained so parser failures remain visible.

In [3]:
# 3. Run the benchmark with a fixed protocol
def extract_choice(text):
    # Match a standalone option letter instead of silently taking any first character.
    match = re.search(r'\b([A-D])\b', text.upper())
    return match.group(1) if match else None

results = []
for item in questions:
    prompt = item['question'] + '\n' + '\n'.join(item['choices']) + '\nReturn only A, B, C, or D.'
    response = client.responses.create(model=TEXT_MODEL, input=prompt, max_output_tokens=1000, reasoning={'effort': 'low'}, store=False)
    raw = response.output_text.strip()
    if not raw:
        # Empty visible output usually means the model exhausted its allowance on reasoning tokens.
        raise RuntimeError(f'No visible answer; status={response.status}, details={response.incomplete_details}')
    prediction = extract_choice(raw)
    results.append({**item, 'prediction': prediction, 'raw': raw, 'correct': prediction == item['answer']})

for row in results:
    print(row['slice'], row['answer'], row['prediction'], 'PASS' if row['correct'] else 'FAIL', '|', row['raw'])

measurement A A PASS | A
measurement B B PASS | B
testing-level B B PASS | B
testing-level A A PASS | A
benchmark-risk A A PASS | A
benchmark-risk A A PASS | A


## 4. Score the benchmark and show uncertainty

A Wilson interval makes the uncertainty from a tiny sample explicit. Slice results show how an overall average can hide concentrated failures.

In [4]:
# 4. Score the benchmark and show uncertainty
def wilson_interval(successes, total, z=1.96):
    # Wilson's interval behaves better than p +/- 1.96*SE for small samples.
    p = successes / total
    denominator = 1 + z**2 / total
    center = (p + z**2 / (2 * total)) / denominator
    margin = z * np.sqrt(p * (1 - p) / total + z**2 / (4 * total**2)) / denominator
    return center - margin, center + margin

correct = sum(row['correct'] for row in results)
lower, upper = wilson_interval(correct, len(results))
print(f'Overall accuracy: {correct}/{len(results)} = {correct / len(results):.1%}')
print(f'Approximate 95% Wilson interval: [{lower:.1%}, {upper:.1%}]')
for slice_name in sorted({row['slice'] for row in results}):
    subset = [row for row in results if row['slice'] == slice_name]
    print(f'{slice_name:16s}: {sum(row["correct"] for row in subset)}/{len(subset)}')

Overall accuracy: 6/6 = 100.0%
Approximate 95% Wilson interval: [61.0%, 100.0%]
benchmark-risk  : 2/2
measurement     : 2/2
testing-level   : 2/2


## 5. Classify model-test intents with embeddings

Prototype descriptions define four bounded testing intents. Cosine similarity assigns each test request to its nearest prototype; the labeled examples let us inspect classification errors.

In [5]:
# 5. Classify model-test intents with embeddings
prototypes = {
    'accuracy': 'Check whether model answers are factually and technically correct against known references.',
    'safety': 'Probe whether model behavior avoids harmful, disallowed, private, or insecure outputs.',
    'robustness': 'Check stability under paraphrases, noise, repeated runs, drift, and irrelevant input changes.',
    'relevance': 'Check whether the response directly addresses the user task without unrelated material.',
}
test_requests = [
    ('accuracy', 'Compare each generated calculation with the independently computed answer.'),
    ('accuracy', 'Verify every citation claim against the supplied source document.'),
    ('safety', 'Try indirect prompt injection and record whether confidential text is exposed.'),
    ('safety', 'Test whether the assistant gives prohibited instructions after a jailbreak attempt.'),
    ('robustness', 'Paraphrase each question five ways and measure prediction consistency.'),
    ('robustness', 'Add harmless typos and compare the resulting labels across repeated trials.'),
    ('relevance', 'Rate whether the answer stays on the requested topic and answers every requested part.'),
    ('relevance', 'Detect answers that are fluent but do not resolve the user question.'),
]
texts = list(prototypes.values()) + [text for _, text in test_requests]
embedding_response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts, dimensions=128)
vectors = np.asarray([item.embedding for item in embedding_response.data], dtype=float)
vectors /= np.linalg.norm(vectors, axis=1, keepdims=True)
prototype_names = list(prototypes)
prototype_vectors = vectors[:len(prototype_names)]
request_vectors = vectors[len(prototype_names):]

embedding_results = []
for (expected, text), vector in zip(test_requests, request_vectors):
    similarities = prototype_vectors @ vector
    predicted = prototype_names[int(np.argmax(similarities))]
    embedding_results.append((expected, predicted, float(np.max(similarities)), text))
    print(f'{expected:10s} -> {predicted:10s} similarity={max(similarities):.3f} | {text}')
print(f'Embedding-classifier accuracy: {np.mean([e == p for e, p, _, _ in embedding_results]):.1%}')

accuracy   -> accuracy   similarity=0.474 | Compare each generated calculation with the independently computed answer.
accuracy   -> accuracy   similarity=0.551 | Verify every citation claim against the supplied source document.
safety     -> safety     similarity=0.543 | Try indirect prompt injection and record whether confidential text is exposed.
safety     -> safety     similarity=0.534 | Test whether the assistant gives prohibited instructions after a jailbreak attempt.
robustness -> robustness similarity=0.447 | Paraphrase each question five ways and measure prediction consistency.
robustness -> robustness similarity=0.530 | Add harmless typos and compare the resulting labels across repeated trials.
relevance  -> relevance  similarity=0.473 | Rate whether the answer stays on the requested topic and answers every requested part.
relevance  -> relevance  similarity=0.473 | Detect answers that are fluent but do not resolve the user question.
Embedding-classifier accuracy: 100.0%


## 6. Interpretation and extensions

The benchmark demonstrates protocol, parsing, uncertainty, and slices; it is too small to compare production models. The embedding classifier demonstrates a semantic instrument, but its prototypes, labels, threshold, model version, and domain shift all need validation. Useful extensions include repeated stochastic trials, held-out cases, human error review, calibrated abstention, and regression tests after application changes.